# Plotting Kothari et al. (2025): The First Half-Century of Empirical Capital Markets Research in Accounting in Pictures

In this exercise, we reproduce one of the plots from [Kothari, et al. (2025, RAST)](https://doi.org/10.1007/s11142-025-09887-3) using Python. Unlike the Sloan (1996) exercise, the goal here isn't to build an empirical analysis from scratch — the authors have generously shared their underlying data, so this exercise is purely about *visualization*: taking someone else's carefully-constructed dataset and turning it into a clean, publication-style chart.

**Background.** "Capital markets research in accounting" (CMRA) is the literature — dating back to Ball and Brown (1968) and Beaver (1968) — that studies how accounting numbers relate to stock prices and returns. Kothari et al. (2025) revisit CMRA's major relations over its first half-century, plotting how each one has evolved over time and relating that time-series variation to market- and macro-level conditions such as inflation and real economic activity.

**Goal of this exercise.** Reproduce the paper's annual CMRA plot: a chosen CMRA statistic (e.g., the median market-adjusted return around "bad news" earnings announcements) plotted by year alongside two macroeconomic series — annual CPI growth and industrial production growth.

**What you'll practice.**
- Reading a multi-sheet Excel workbook with a non-standard header row.
- Cleaning a text-coded missing-value convention (`"."` for missing) and coercing columns to numeric.
- Filtering long-format data by category columns (`VarName`, `Sample`) and merging it with a second table on a shared key (`YEAR`).
- Building a multi-series `matplotlib` line chart with markers, a reference line, and a legend styled to resemble a published figure.

**Data.** All data are shared publicly by the authors at <https://www.colorado.edu/faculty/schonberger>. This exercise focuses purely on presentation — the underlying numbers are already computed for you.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Table header is on row 3 (0-indexed header=2)
cmra: pd.DataFrame = pd.read_excel("../data/Kothari_2025_data.xlsx", sheet_name="CMRA Database - Annual", header=2)

In [ ]:
cmra = cmra[["PlotNumber", "Sample", "VarName", "YEAR", "StatValue"]].copy()

ts: pd.DataFrame = pd.read_excel("../data/Kothari_2025_data.xlsx", sheet_name="TS Dets Data - Ann", header=2)
ts.rename(columns={"year": "YEAR"}, inplace=True)

# Our raw data stores missing macro values as the text "." -> convert to NaN,
# and make sure every macro column is numeric.
macro_cols: list[str] = [c for c in ts.columns if c != "YEAR"]
for c in macro_cols:
    ts[c] = pd.to_numeric(ts[c].replace(".", pd.NA), errors="coerce")

In [ ]:
def plot_cmra(variable: str, 
              sample: str) -> tuple[plt.Figure, plt.Axes]:
    """
    Draw the line chart: YEAR (x-axis) vs. selected CMRA variable, CPI
    growth, and industrial production growth.
    """

    _temp: pd.DataFrame = cmra.loc[(cmra["VarName"] == variable) & (cmra["Sample"] == sample)].copy()
    if _temp.empty:
        raise ValueError(f"No rows found for VarName='{variable}' and Sample='{sample}'.")
    df: pd.DataFrame = _temp.merge(ts, on="YEAR", how="left").sort_values("YEAR")
    
    fig, ax = plt.subplots(figsize=(11, 6))
    
    ax.plot(df["YEAR"], df["StatValue"], 
            marker="o", markersize=5, 
            color="blue", alpha=1.0,
            linewidth=1.5, label=variable)
    ax.plot(df["YEAR"], df["CPI_Growth_Annual"], 
            marker="x", markersize=3, 
            color="green", alpha=0.7,
            linewidth=1.5, label="CPI_Growth_Annual")
    ax.plot(df["YEAR"], df["IndProdGrowth_Annual"], 
            marker="v", markersize=3, 
            color="red", alpha=0.7,
            linewidth=1.5, label="IndProdGrowth_Annual")

    ax.set_title("Annual CMRA Plot", fontsize=12, fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel(f"{variable}  /  Macro growth rate")
    ax.axhline(0, color="black", alpha=0.5, linewidth=0.8)
    ax.legend(loc="lower center", fontsize=10)
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    return fig, ax

In [ ]:
print(list(cmra['VarName'].unique()))

In [ ]:
fig, ax = plot_cmra(variable = "BadNews_MARET_Median", sample = "Full")
plt.show()